# **Project Name** - Uber Supply-Demand Gap Analysis

##### **Project Type** - EDA
##### **Contribution** - Individual

# **Project Workflow Overview**

This project follows the **4-tool methodology** as required:

| Step | Tool | What we do |
|------|------|-----------|
| **Step 1** | Microsoft Excel | Data cleaning — fix timestamps, handle missing values, add derived columns, build summary tables & dashboard |
| **Step 2** | SQL (SQLite) | Run 12 business insight queries to answer specific problem statements |
| **Step 3** | Python / Pandas | Full EDA with 15+ visualizations and statistical analysis |
| **Step 4** | This Notebook | Combines all findings into a complete project narrative |

> **Files in this project:**
> - `Uber Request Data.csv` — Original raw data
> - `Uber_EDA_Excel.xlsx` — Cleaned data + Summary Tables + Dashboard (Excel)
> - `Uber_SQL_Insights.sql` — 12 SQL queries with business insights
> - `Uber_Supply_Demand_Gap_EDA_Solution.ipynb` — This Pandas EDA notebook


# **Project Summary -**

This project performs an in-depth Exploratory Data Analysis (EDA) on Uber ride request data collected over a week in July 2016. The dataset contains 6,745 ride requests with details about pickup points (City or Airport), driver assignment status, and timestamps.

The primary goal is to identify the **supply-demand gap** — i.e., times and locations where Uber struggles to fulfill ride requests either due to unavailability of cars or driver cancellations. Understanding this gap helps Uber optimize driver allocation, reduce customer dissatisfaction, and increase revenue.

Key findings include: a significant morning rush (5–10 AM) from the City where drivers frequently cancel, an evening rush (5–10 PM) from the Airport where no cars are available, and an overall fulfillment rate of only ~42%. These patterns point to actionable strategies for driver incentivization and demand-based fleet management.

# **Problem Statement**

Uber is facing a significant supply-demand mismatch for cab requests between the City and the Airport. Many requests are either cancelled by drivers or cannot be fulfilled due to unavailability of cars. The business objective is to identify the root causes of this gap and suggest actionable strategies to resolve it.

#### **Business Objective:** Identify the time slots and trip routes where the supply-demand gap is most severe, and provide data-driven recommendations to bridge this gap.

# ***Let's Begin!***
## ***1. Know Your Data***
### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set global plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
print("Libraries imported successfully.")

### Dataset Loading

In [ ]:
# Load Dataset
df = pd.read_csv('Uber Request Data.csv')
print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")

### Dataset First View

In [ ]:
# Dataset First Look
df.head(10)

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f"Number of Rows    : {df.shape[0]}")
print(f"Number of Columns : {df.shape[1]}")

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print(f"Number of duplicate rows: {df.duplicated().sum()}")

#### Missing Values / Null Values

In [ ]:
# Missing Values/Null Values Count
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(8, 4))
missing = df.isnull().sum()
missing = missing[missing > 0]
missing.plot(kind='bar', color=['#e74c3c', '#e67e22'], edgecolor='black')
plt.title('Missing Values per Column', fontsize=14)
plt.ylabel('Count')
plt.xticks(rotation=0)
for i, v in enumerate(missing):
    plt.text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### What did you know about your dataset?

- The dataset has **6,745 rows and 6 columns**.
- `Driver id` has **2,650 missing values** — these correspond to requests where 'No Cars Available' (no driver was assigned).
- `Drop timestamp` has **3,914 missing values** — these represent trips that were either Cancelled or had no car available (so no drop time exists).
- There are **no duplicate rows**.
- The `Request timestamp` column has two inconsistent date formats that need cleaning.
- Status has 3 categories: *Trip Completed*, *Cancelled*, and *No Cars Available*.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print("Column Names:")
for col in df.columns:
    print(f"  - {col}")

In [ ]:
# Dataset Describe
df.describe(include='all')

### Variables Description

| Column | Type | Description |
|--------|------|-------------|
| `Request id` | int | Unique identifier for each ride request |
| `Pickup point` | str | Origin of the trip — either **City** or **Airport** |
| `Driver id` | float | Unique driver identifier; NaN if no driver assigned |
| `Status` | str | Outcome: *Trip Completed*, *Cancelled*, or *No Cars Available* |
| `Request timestamp` | str | Date and time the ride was requested |
| `Drop timestamp` | str | Date and time of trip completion; NaN if not completed |

In [ ]:
# Check Unique Values for each variable
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")
    if df[col].nunique() <= 5:
        print(f"  Values: {df[col].unique().tolist()}")

## 3. ***Data Wrangling***
### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

# --- Step 1: Parse mixed-format timestamps ---
def parse_ts(ts):
    if pd.isna(ts):
        return pd.NaT
    for fmt in ('%d/%m/%Y %H:%M', '%d-%m-%Y %H:%M:%S'):
        try:
            return pd.to_datetime(ts, format=fmt)
        except:
            pass
    return pd.NaT

df['Request timestamp'] = df['Request timestamp'].apply(parse_ts)
df['Drop timestamp']    = df['Drop timestamp'].apply(parse_ts)

# --- Step 2: Extract time features ---
df['Request_Hour']    = df['Request timestamp'].dt.hour
df['Request_Day']     = df['Request timestamp'].dt.day
df['Request_Weekday'] = df['Request timestamp'].dt.day_name()

# --- Step 3: Define time-of-day slots ---
def time_slot(hour):
    if 5 <= hour < 10:
        return 'Morning Rush (5-10)'
    elif 10 <= hour < 17:
        return 'Daytime (10-17)'
    elif 17 <= hour < 22:
        return 'Evening Rush (17-22)'
    else:
        return 'Late Night (22-5)'

df['Time_Slot'] = df['Request_Hour'].apply(time_slot)

# --- Step 4: Trip duration (for completed trips) ---
df['Trip_Duration_min'] = (df['Drop timestamp'] - df['Request timestamp']).dt.total_seconds() / 60

# --- Step 5: Binary flag for fulfilled requests ---
df['Is_Fulfilled'] = (df['Status'] == 'Trip Completed').astype(int)

print("Data wrangling complete. New columns added:")
print(df[['Request_Hour','Time_Slot','Trip_Duration_min','Is_Fulfilled']].head(5))

### What all manipulations have you done and insights you found?

1. **Timestamp Parsing**: The `Request timestamp` column had two date formats (`dd/mm/yyyy HH:MM` and `dd-mm-yyyy HH:MM:SS`). A custom parser was applied to standardize both.
2. **Feature Engineering**: Extracted `Request_Hour`, `Request_Day`, and `Request_Weekday` for time-based analysis.
3. **Time Slot Bucketing**: Created a `Time_Slot` column grouping hours into Morning Rush, Daytime, Evening Rush, and Late Night for cleaner visualizations.
4. **Trip Duration**: Calculated trip duration in minutes for completed trips — useful for identifying trip length patterns.
5. **Is_Fulfilled Flag**: Binary column (1 = Trip Completed, 0 = not) to easily compute fulfillment rates.

## **SQL Insights Summary** (from `Uber_SQL_Insights.sql`)

Before running the Pandas visualizations, here are the key findings already confirmed via SQL queries:

| SQL Query | Finding |
|-----------|---------|
| PS-1: Status Distribution | Only **42.0%** fulfilled; 39.3% No Cars, 18.7% Cancelled |
| PS-2: Status by Pickup | City: 39% Cancellations; Airport: 60% No Cars Available |
| PS-3: Hourly Demand | Peak hours: **5–10 AM** and **5–10 PM** |
| PS-4: City Cancellation Rate | **50%+ cancellation rate** between 5–9 AM from City |
| PS-5: Airport No-Cars Rate | **70%+ no-cars rate** between 17–21 PM at Airport |
| PS-6: Time Slot Gap | Evening Rush has the **largest absolute gap (1,197 unfulfilled)** |
| PS-7: Driver Capacity | ~300 unique drivers; avg ~9.4 completed trips/driver |
| PS-8: Trip Duration | Airport trips **~52 min avg** vs City **~47 min avg** |
| PS-9: Day-wise | Demand is **consistent across weekdays** — not day-specific |
| PS-10: Cross-tab | Airport + Evening Rush + No Cars = **1,048** and City + Morning Rush + Cancelled = **786** |
| PS-11: Critical Windows | Just 2 problem windows = **~27% of all requests unfulfilled** |
| PS-12: Hour Gap Rank | Hours 8–9 AM and 18–19 PM have the **highest absolute supply gap** |

These SQL-confirmed findings are now visualized in detail in the charts below.


### Running SQL Queries in Python (via SQLite)

In [ ]:
# Running SQL queries directly inside the notebook using SQLite
import sqlite3
import pandas as pd

# Load cleaned data into an in-memory SQLite database
df_sql = pd.read_csv('Uber Request Data.csv')

def parse_ts(ts):
    if pd.isna(ts): return pd.NaT
    for fmt in ('%d/%m/%Y %H:%M', '%d-%m-%Y %H:%M:%S'):
        try: return pd.to_datetime(ts, format=fmt)
        except: pass
    return pd.NaT

df_sql['Request timestamp'] = df_sql['Request timestamp'].apply(parse_ts)
df_sql['Drop timestamp']    = df_sql['Drop timestamp'].apply(parse_ts)
df_sql['Request_Hour']      = df_sql['Request timestamp'].dt.hour
df_sql['Request_Weekday']   = df_sql['Request timestamp'].dt.day_name()
df_sql['Trip_Duration_min'] = (df_sql['Drop timestamp'] - df_sql['Request timestamp']).dt.total_seconds() / 60

def time_slot(h):
    if 5 <= h < 10:    return 'Morning Rush (5-10)'
    elif 10 <= h < 17: return 'Daytime (10-17)'
    elif 17 <= h < 22: return 'Evening Rush (17-22)'
    else:              return 'Late Night (22-5)'

df_sql['Time_Slot']    = df_sql['Request_Hour'].apply(time_slot)
df_sql['Is_Fulfilled'] = (df_sql['Status'] == 'Trip Completed').astype(int)

# Rename columns to match SQL schema
df_sql.columns = [c.lower().replace(' ', '_') for c in df_sql.columns]

conn = sqlite3.connect(':memory:')
df_sql.to_sql('uber_requests', conn, index=False, if_exists='replace')
print("SQLite DB ready with", pd.read_sql("SELECT COUNT(*) as rows FROM uber_requests", conn).iloc[0,0], "rows")

In [ ]:
# SQL — Problem Statement 1: Overall Status Distribution
query1 = '''
SELECT
    status,
    COUNT(*) AS total_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM uber_requests), 2) AS percentage
FROM uber_requests
GROUP BY status
ORDER BY total_count DESC
'''
print("PS-1: Overall Status Distribution")
print(pd.read_sql(query1, conn).to_string(index=False))

In [ ]:
# SQL — Problem Statement 2: Status by Pickup Point
query2 = '''
SELECT
    pickup_point,
    status,
    COUNT(*) AS count
FROM uber_requests
GROUP BY pickup_point, status
ORDER BY pickup_point, count DESC
'''
print("PS-2: Status by Pickup Point")
print(pd.read_sql(query2, conn).to_string(index=False))

In [ ]:
# SQL — Problem Statement 3 & 4: Cancellation Rate by Hour (City)
query3 = '''
SELECT
    request_hour,
    COUNT(*) AS total_city_requests,
    SUM(CASE WHEN status = "Cancelled" THEN 1 ELSE 0 END) AS cancellations,
    ROUND(SUM(CASE WHEN status = "Cancelled" THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS cancel_rate_pct
FROM uber_requests
WHERE pickup_point = "City"
GROUP BY request_hour
ORDER BY cancel_rate_pct DESC
LIMIT 8
'''
print("PS-4: Top Hours with Highest Cancellation Rate (City Pickups)")
print(pd.read_sql(query3, conn).to_string(index=False))

In [ ]:
# SQL — Problem Statement 5: No Cars Rate at Airport by Hour
query4 = '''
SELECT
    request_hour,
    COUNT(*) AS total_airport_requests,
    SUM(CASE WHEN status = "No Cars Available" THEN 1 ELSE 0 END) AS no_cars_count,
    ROUND(SUM(CASE WHEN status = "No Cars Available" THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS no_cars_rate_pct
FROM uber_requests
WHERE pickup_point = "Airport"
GROUP BY request_hour
ORDER BY no_cars_rate_pct DESC
LIMIT 8
'''
print("PS-5: Top Hours with Highest No-Cars Rate (Airport Pickups)")
print(pd.read_sql(query4, conn).to_string(index=False))

In [ ]:
# SQL — Problem Statement 11: The Two Critical Problem Windows
query5 = '''
SELECT "City Morning Rush Cancellations" AS problem_window,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM uber_requests), 2) AS pct_of_total
FROM uber_requests
WHERE pickup_point = "City"
  AND time_slot = "Morning Rush (5-10)"
  AND status = "Cancelled"
UNION ALL
SELECT "Airport Evening Rush No Cars" AS problem_window,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM uber_requests), 2) AS pct_of_total
FROM uber_requests
WHERE pickup_point = "Airport"
  AND time_slot = "Evening Rush (17-22)"
  AND status = "No Cars Available"
'''
print("PS-11: The Two Critical Problem Windows")
result = pd.read_sql(query5, conn)
print(result.to_string(index=False))
total_pct = result['pct_of_total'].sum()
print(f"\nCombined: {total_pct:.1f}% of ALL requests are from just these 2 problem windows!")

## ***4. Data Visualization, Storytelling & Experimenting with Charts***

#### Chart - 1

In [ ]:
# Chart - 1: Status Distribution (Pie + Bar)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

status_counts = df['Status'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#e67e22']

# Pie
axes[0].pie(status_counts, labels=status_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=140, wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[0].set_title('Proportion of Request Statuses')

# Bar
status_counts.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_title('Count of Request Statuses')
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Number of Requests')
axes[1].tick_params(axis='x', rotation=15)
for p in axes[1].patches:
    axes[1].annotate(str(int(p.get_height())),
                     (p.get_x() + p.get_width()/2, p.get_height() + 30),
                     ha='center', fontweight='bold')

plt.suptitle('Overall Uber Request Status', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A pie chart quickly shows proportional split, while a bar chart gives exact counts. Together they provide both relative and absolute perspectives on request outcomes.

##### 2. What is/are the insight(s) found from the chart?
Only **42% of requests result in a completed trip**. A combined **58% are either cancelled (18.7%) or have no cars available (39.3%)**. This is the core supply-demand gap — the majority of demand is unfulfilled.

##### 3. Will the gained insights help creating a positive business impact?
Yes. This directly quantifies revenue loss. Nearly 4,000 unfulfilled trips represent lost revenue and customer churn. Resolving even half of these would significantly increase Uber's income. The 'No Cars Available' category is the bigger problem and should be the primary focus.

#### Chart - 2

In [ ]:
# Chart - 2: Status by Pickup Point (Stacked Bar)
ct = pd.crosstab(df['Pickup point'], df['Status'])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

ct_pct.plot(kind='bar', stacked=True, figsize=(9, 5),
            color=['#e67e22', '#e74c3c', '#2ecc71'], edgecolor='white')
plt.title('Request Status Distribution by Pickup Point (%)', fontsize=14)
plt.xlabel('Pickup Point')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Status', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

print(ct)

##### 1. Why did you pick the specific chart?
A 100% stacked bar chart normalizes counts and lets us compare the status composition between City and Airport on equal footing.

##### 2. What is/are the insight(s) found from the chart?
- From **City**: ~39% of trips are **Cancelled** by drivers — likely because drivers don't want to head to the airport empty-handed.
- From **Airport**: ~60% face **No Cars Available** — drivers are already at the city after dropping off, creating a vacuum at the airport.
- This asymmetry is the root cause of the supply-demand gap.

##### 3. Business Impact?
Addressing this directional imbalance (city-to-airport morning rush causing cancellations; airport-to-city evening gap causing no availability) is the most impactful fix. Incentivizing drivers to pick up from the Airport in the evening could significantly improve metrics.

#### Chart - 3

In [ ]:
# Chart - 3: Hourly Demand (Request Volume by Hour)
plt.figure(figsize=(12, 5))
hourly = df.groupby('Request_Hour').size().reset_index(name='Requests')
sns.lineplot(data=hourly, x='Request_Hour', y='Requests', marker='o', linewidth=2.5, color='#2980b9')
plt.fill_between(hourly['Request_Hour'], hourly['Requests'], alpha=0.15, color='#2980b9')
plt.axvspan(5, 9.5, alpha=0.1, color='red', label='Morning Rush')
plt.axvspan(17, 21.5, alpha=0.1, color='orange', label='Evening Rush')
plt.title('Hourly Distribution of Ride Requests', fontsize=14)
plt.xlabel('Hour of Day')
plt.ylabel('Number of Requests')
plt.xticks(range(0, 24))
plt.legend()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A line chart with area fill is ideal for showing demand fluctuation over continuous time (hours).

##### 2. Insights?
Two clear demand spikes: a **morning rush (5–10 AM)** and an **evening rush (5–10 PM)**. Demand is very low from midnight to 4 AM. These are the critical windows for supply planning.

##### 3. Business Impact?
Uber should implement **surge pricing** or **driver incentives** specifically during these two rush windows to pull more drivers online. Predictive notifications to drivers before rush hours can also pre-position supply.

#### Chart - 4

In [ ]:
# Chart - 4: Heatmap - Hour vs Status Count
pivot = df.pivot_table(index='Status', columns='Request_Hour', aggfunc='size', fill_value=0)
plt.figure(figsize=(16, 4))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, annot=True, fmt='d', annot_kws={'size': 7})
plt.title('Request Count by Status and Hour of Day', fontsize=14)
plt.xlabel('Hour of Day')
plt.ylabel('Status')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A heatmap simultaneously shows how all three status types evolve across all 24 hours, making time-based patterns immediately visible.

##### 2. Insights?
- **Cancellations** peak sharply between **5–10 AM**.
- **No Cars Available** peaks between **17–22 PM**.
- **Trip Completions** are relatively spread but peak in morning and evening.

##### 3. Business Impact?
This precise hour-by-hour breakdown helps Uber set **time-targeted driver bonuses** — morning bonuses to reduce cancellations, and evening bonuses to attract more drivers to the Airport area.

#### Chart - 5

In [ ]:
# Chart - 5: Cancellations by Pickup Point and Hour
cancelled = df[df['Status'] == 'Cancelled']
no_cars   = df[df['Status'] == 'No Cars Available']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, sub, title, color in zip(
    axes,
    [cancelled, no_cars],
    ['Cancellations by Hour & Pickup', 'No Cars Available by Hour & Pickup'],
    ['#e74c3c', '#e67e22']
):
    for pt, ls in [('City', '-'), ('Airport', '--')]:
        data = sub[sub['Pickup point'] == pt].groupby('Request_Hour').size()
        ax.plot(data.index, data.values, marker='o', linestyle=ls, label=pt, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Count')
    ax.set_xticks(range(0, 24))
    ax.legend()
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Two side-by-side line charts comparing City vs Airport for the two problem statuses, enabling direct visual comparison of when and where each problem occurs.

##### 2. Insights?
- **Cancellations** are almost entirely a **City problem in the morning** — drivers at the city don't want to make the airport trip early.
- **No Cars Available** is almost entirely an **Airport problem in the evening** — after morning drop-offs at the city, cars don't return to the airport.

##### 3. Business Impact?
This confirms the directional imbalance. A **shuttle/incentive program** for drivers completing airport drop-offs in the morning to stay and pick up return flights would reduce both problems simultaneously.

#### Chart - 6

In [ ]:
# Chart - 6: Time Slot vs Status (Grouped Bar)
slot_order = ['Morning Rush (5-10)', 'Daytime (10-17)', 'Evening Rush (17-22)', 'Late Night (22-5)']
ct2 = pd.crosstab(df['Time_Slot'], df['Status']).reindex(slot_order)

ct2.plot(kind='bar', figsize=(12, 5), color=['#e67e22','#e74c3c','#2ecc71'], edgecolor='black')
plt.title('Request Status by Time Slot', fontsize=14)
plt.xlabel('Time Slot')
plt.ylabel('Number of Requests')
plt.xticks(rotation=15)
plt.legend(title='Status')
for p in plt.gca().patches:
    if p.get_height() > 0:
        plt.gca().annotate(str(int(p.get_height())),
                           (p.get_x() + p.get_width()/2, p.get_height() + 10),
                           ha='center', fontsize=8)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Grouped bar chart allows direct count comparison across time slots for each status type.

##### 2. Insights?
- Morning Rush has the **most cancellations (889)**.
- Evening Rush has the **most 'No Cars Available' (1,197)**.
- Daytime and Late Night are relatively stable.

##### 3. Business Impact?
Time-slot-based driver incentive tiers would be far more cost-effective than flat incentives, targeting only the two problem windows.

#### Chart - 7

In [ ]:
# Chart - 7: Fulfillment Rate by Pickup Point and Time Slot
fulfil = df.groupby(['Pickup point', 'Time_Slot'])['Is_Fulfilled'].mean().reset_index()
fulfil['Fulfillment Rate (%)'] = fulfil['Is_Fulfilled'] * 100
fulfil['Time_Slot'] = pd.Categorical(fulfil['Time_Slot'], categories=slot_order, ordered=True)
fulfil = fulfil.sort_values('Time_Slot')

plt.figure(figsize=(12, 5))
sns.barplot(data=fulfil, x='Time_Slot', y='Fulfillment Rate (%)', hue='Pickup point',
            palette=['#3498db', '#e74c3c'])
plt.title('Fulfillment Rate (%) by Pickup Point & Time Slot', fontsize=14)
plt.xlabel('Time Slot')
plt.ylabel('Fulfillment Rate (%)')
plt.ylim(0, 100)
plt.axhline(50, color='grey', linestyle='--', label='50% line')
plt.legend()
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A grouped bar chart showing fulfillment rates (%) gives a normalized view that is easy to benchmark against a 50% or 100% target.

##### 2. Insights?
- **City Morning Rush**: Fulfillment rate is extremely low (~20%) due to cancellations.
- **Airport Evening Rush**: Fulfillment rate is the lowest (~20%) due to no cars.
- **Daytime from both points**: Highest fulfillment (~65–70%).

##### 3. Business Impact?
This quantifies the service failure rate. These two slots together account for the bulk of customer dissatisfaction and are the clearest targets for operational improvement.

#### Chart - 8

In [ ]:
# Chart - 8: Trip Duration Distribution (Box + Violin)
completed = df[df['Status'] == 'Trip Completed'].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=completed, x='Pickup point', y='Trip_Duration_min',
            palette='Set2', ax=axes[0])
axes[0].set_title('Trip Duration by Pickup Point (Box)')
axes[0].set_ylabel('Duration (minutes)')

sns.violinplot(data=completed, x='Pickup point', y='Trip_Duration_min',
               palette='Set2', ax=axes[1], inner='quartile')
axes[1].set_title('Trip Duration by Pickup Point (Violin)')
axes[1].set_ylabel('Duration (minutes)')

plt.suptitle('Distribution of Completed Trip Durations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(completed.groupby('Pickup point')['Trip_Duration_min'].describe().round(1))

##### 1. Why did you pick the specific chart?
Box + Violin plots together show both the statistical summary (quartiles, outliers) and the full distribution shape — making them superior to a simple bar chart for duration analysis.

##### 2. Insights?
- Airport trips are on average **longer** (median ~52 min) than City trips (~47 min), which makes sense geographically.
- Both distributions are fairly normal with a few longer outliers.

##### 3. Business Impact?
Longer Airport trips mean higher per-trip revenue but also slower driver turnaround. This further explains why fewer drivers volunteer for Airport pickups during peak hours — they get 'stuck' farther from the city.

#### Chart - 9

In [ ]:
# Chart - 9: Day-wise Request Volume
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df.groupby('Request_Weekday').size().reindex(day_order)

plt.figure(figsize=(10, 5))
bars = plt.bar(day_counts.index, day_counts.values,
               color=sns.color_palette('viridis', 7), edgecolor='black')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
             str(bar.get_height()), ha='center', fontweight='bold')
plt.title('Total Ride Requests by Day of Week', fontsize=14)
plt.xlabel('Day')
plt.ylabel('Number of Requests')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A simple bar chart ordered by day of week is the clearest way to compare weekday demand patterns.

##### 2. Insights?
Requests are relatively uniform across weekdays (the data spans weekdays only, Mon–Fri). There's a slight mid-week peak. No clear weekend effect in this dataset window.

##### 3. Business Impact?
Suggests consistent daily demand — no major day-specific surge events during this week. Operational planning should be time-of-day focused rather than day-of-week focused.

#### Chart - 10

In [ ]:
# Chart - 10: Supply-Demand Gap Bar (by Hour)
hourly_status = df.groupby(['Request_Hour', 'Status']).size().unstack(fill_value=0)
hourly_status['Demand'] = hourly_status.sum(axis=1)
hourly_status['Supply'] = hourly_status.get('Trip Completed', 0)
hourly_status['Gap'] = hourly_status['Demand'] - hourly_status['Supply']

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(hourly_status.index, hourly_status['Demand'], label='Total Demand', color='#3498db', alpha=0.7)
ax.bar(hourly_status.index, hourly_status['Supply'], label='Fulfilled (Supply)', color='#2ecc71', alpha=0.9)
ax.plot(hourly_status.index, hourly_status['Gap'], color='red', marker='D',
        linewidth=2, label='Gap (Unfulfilled)')
ax.set_title('Supply vs Demand Gap by Hour', fontsize=14)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Requests')
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Overlapping bars + a line for the gap directly visualizes the core business problem — supply shortfall at each hour.

##### 2. Insights?
The gap peaks at **Hours 8–9 (morning)** and **Hours 18–21 (evening)**. These are the two critical supply deficit windows.

##### 3. Business Impact?
This chart is the core deliverable of the analysis. It directly answers the business question and provides a clear, hour-by-hour profile of where operational investment is needed.

#### Chart - 11

In [ ]:
# Chart - 11: Cancellation Rate by Hour (City only)
city_df = df[df['Pickup point'] == 'City']
city_hourly = city_df.groupby('Request_Hour')['Status'].value_counts().unstack(fill_value=0)
city_hourly['Cancel_Rate'] = city_hourly.get('Cancelled', 0) / city_hourly.sum(axis=1) * 100

plt.figure(figsize=(12, 5))
plt.fill_between(city_hourly.index, city_hourly['Cancel_Rate'],
                 alpha=0.4, color='#e74c3c')
plt.plot(city_hourly.index, city_hourly['Cancel_Rate'],
         color='#e74c3c', marker='o', linewidth=2)
plt.axhspan(0, 10, alpha=0.05, color='green', label='Acceptable (<10%)')
plt.title('Driver Cancellation Rate by Hour (City Pickups)', fontsize=14)
plt.xlabel('Hour of Day')
plt.ylabel('Cancellation Rate (%)')
plt.xticks(range(0, 24))
plt.legend()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
An area chart emphasizes the severity of the cancellation problem over time, with a reference band for 'acceptable' cancellation rate.

##### 2. Insights?
City cancellation rates exceed **50% during 5–9 AM** — i.e., more than half of morning City requests are cancelled. This is a critical service failure.

##### 3. Business Impact?
Drivers cancelling morning City pickups likely do so to avoid the long airport run. A guaranteed return-trip incentive or airport queue system could reduce this dramatically.

#### Chart - 12

In [ ]:
# Chart - 12: No-Cars Rate at Airport by Hour
airport_df = df[df['Pickup point'] == 'Airport']
ap_hourly = airport_df.groupby('Request_Hour')['Status'].value_counts().unstack(fill_value=0)
ap_hourly['NoCar_Rate'] = ap_hourly.get('No Cars Available', 0) / ap_hourly.sum(axis=1) * 100

plt.figure(figsize=(12, 5))
plt.fill_between(ap_hourly.index, ap_hourly['NoCar_Rate'],
                 alpha=0.4, color='#e67e22')
plt.plot(ap_hourly.index, ap_hourly['NoCar_Rate'],
         color='#e67e22', marker='s', linewidth=2)
plt.title('No Cars Available Rate by Hour (Airport Pickups)', fontsize=14)
plt.xlabel('Hour of Day')
plt.ylabel('No Cars Rate (%)')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Mirrors Chart 11 for Airport / No Cars scenario, enabling side-by-side comparison of the two major problem patterns.

##### 2. Insights?
No-cars rate at Airport exceeds **70% during 17–21 PM** — an extremely severe shortage of supply at the airport during the evening rush.

##### 3. Business Impact?
This quantifies the evening airport gap. If Uber could ensure even 30% more drivers are at the airport between 5–9 PM, customer experience would improve substantially.

#### Chart - 13

In [ ]:
# Chart - 13: Multivariate - Pickup Point x Time Slot x Status (Heatmap)
multi = df.groupby(['Pickup point', 'Time_Slot', 'Status']).size().reset_index(name='Count')
multi['Time_Slot'] = pd.Categorical(multi['Time_Slot'], categories=slot_order, ordered=True)

pivot3 = multi.pivot_table(index=['Pickup point', 'Status'], columns='Time_Slot', values='Count', fill_value=0)
pivot3 = pivot3[slot_order]

plt.figure(figsize=(14, 7))
sns.heatmap(pivot3, annot=True, fmt='d', cmap='Blues', linewidths=0.4, annot_kws={'size': 10})
plt.title('Request Count: Pickup Point × Status × Time Slot', fontsize=14)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A multi-index heatmap is an efficient multivariate display — it simultaneously shows pickup point, status type, and time slot in one view.

##### 2. Insights?
The darkest cells (most volume) are: **City + Cancelled + Morning Rush (786)** and **Airport + No Cars + Evening Rush (1,048)**. These two cells represent the core of the supply-demand gap.

##### 3. Business Impact?
Focusing on just these two problem cells — morning City cancellations and evening Airport no-car situations — would resolve the majority of service failures.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
corr_df = df[['Request_Hour', 'Is_Fulfilled', 'Trip_Duration_min', 'Driver id']].copy()
corr_df['Has_Driver'] = df['Driver id'].notna().astype(int)
corr_df = corr_df.drop(columns='Driver id')
corr_df['Is_Airport'] = (df['Pickup point'] == 'Airport').astype(int)

plt.figure(figsize=(7, 5))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A correlation heatmap surfaces linear relationships between all numerical variables at once, useful for identifying driving factors.

##### 2. Insights?
- `Has_Driver` and `Is_Fulfilled` are perfectly correlated — obviously, a trip can't be completed without a driver.
- `Request_Hour` has a moderate negative correlation with `Is_Fulfilled` — later evening hours are harder to fulfill.
- `Trip_Duration_min` correlates with `Is_Airport` (Airport trips are longer).

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
pair_df = df[df['Status'] == 'Trip Completed'][['Request_Hour', 'Trip_Duration_min']].copy()
pair_df['Pickup point'] = df[df['Status'] == 'Trip Completed']['Pickup point']

sns.pairplot(pair_df, hue='Pickup point', palette={'City':'#3498db', 'Airport':'#e74c3c'},
             plot_kws={'alpha': 0.5}, diag_kind='kde')
plt.suptitle('Pair Plot: Completed Trips by Pickup Point', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A pair plot provides a grid of bivariate scatter plots and univariate KDEs, colored by pickup point, to explore all pairwise relationships together.

##### 2. Insights?
City and Airport trips show distinct clusters in the `Request_Hour` vs `Trip_Duration_min` space. Airport trips are more uniformly spread across hours while City trips concentrate around rush hours. Duration distributions overlap substantially, confirming geographic similarity.

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective?

Based on the EDA, the following targeted recommendations address the supply-demand gap:

**1. Morning Rush — City to Airport (Cancellation Problem)**
- Introduce **guaranteed trip bonuses** for drivers who complete a City→Airport ride between 5–10 AM.
- Implement a **pre-booking system** where airport-bound passengers book 30+ minutes ahead, reducing driver uncertainty and cancellations.
- Show drivers the **estimated earnings** for the airport trip before they accept, incentivizing acceptance.

**2. Evening Rush — Airport to City (No Cars Problem)**
- Create an **Airport Driver Pool**: offer arriving drivers a +20% fare incentive to position themselves at the airport between 5–10 PM.
- Enable **dynamic surge pricing** at the Airport in the evening to attract more drivers organically.
- Partner with airlines to get **flight arrival data**, predicting demand spikes and pre-positioning drivers.

**3. Systemic Improvements**
- Launch a **Driver Loyalty Program** rewarding drivers who maintain low cancellation rates and high airport service.
- Use **real-time demand forecasting** to send push notifications to offline drivers when a surge is imminent.
- Consider a **dedicated airport fleet** (small pool of drivers exclusively serving airport routes) to guarantee minimum availability.

These measures, if implemented, could potentially increase trip fulfillment from ~42% to 65–70%, representing a ~60% increase in completed rides and revenue.


# **Conclusion**

This EDA of the Uber Request dataset reveals a clear and structural **supply-demand gap** with two distinct patterns:

1. **Morning Problem (5–10 AM, City → Airport)**: High cancellation rates (~50%+) from City drivers who are unwilling to make the long airport trip during peak hours, leading to ~786 unfulfilled requests in just this window.

2. **Evening Problem (5–10 PM, Airport → City)**: Severe car unavailability (~70%+) at the airport during evening hours, resulting in ~1,048 requests with no cars available.

Together, these two problem windows account for the majority of all unfulfilled rides in the dataset. The overall fulfillment rate stands at only **~42%**, meaning **58% of customer demand goes unmet**.

Key takeaways:
- The gap is **temporal and directional** — not a general shortage but a location+time specific imbalance.
- **Driver behavior** (cancellations) and **fleet positioning** (cars not at airport in evening) are the two root causes.
- Data-driven interventions — targeted incentives, surge pricing, pre-positioning, and demand forecasting — can realistically close a significant portion of this gap.

This analysis provides Uber with a clear roadmap to prioritize operational improvements and maximize both customer satisfaction and revenue.

### ***Hurrah! You have successfully completed your EDA Capstone Project!!!***
